## Motivation

The lab needed a way to stream capacitive sensor data over BLE to a REST API without blocking
the main thread. Off-the-shelf solutions were either synchronous (bad for long-running reads)
or didn't expose a clean HTTP interface for downstream consumers.

LabSense solves this with a fully async stack: `bleak` handles BLE notifications on an asyncio
event loop, `FastAPI` exposes the data over HTTP, and `aiosqlite` persists readings without
blocking either layer.

> **Key constraint:** BLE GATT notifications are push-based. The sensor decides when to send.
> A synchronous read loop would miss packets under load. Hence: full async.


## Architecture

```
┌─────────────────┐     BLE GATT      ┌──────────────────┐
│  Sensor Array   │ ────────────────► │  bleak client    │
│  (capacitive)   │   notifications   │  (asyncio loop)  │
└─────────────────┘                   └────────┬─────────┘
                                               │ asyncio.Queue
                                    ┌──────────▼──────────┐
                                    │   FastAPI router    │
                                    │   /stream  /latest  │
                                    └──────────┬──────────┘
                                               │
                                    ┌──────────▼──────────┐
                                    │   aiosqlite db      │
                                    │   readings.db       │
                                    └─────────────────────┘
```


## Signal Model

The capacitive sensor outputs a normalized admittance value $\hat{Y}$ per channel. For an
array of $N$ electrodes, the raw reading vector is:

$$
\mathbf{y}[t] = \begin{bmatrix} \hat{Y}_1[t] \\ \vdots \\ \hat{Y}_N[t] \end{bmatrix}
\in \mathbb{R}^N
$$

The drift component $\mathbf{d}[t]$ is modeled as a slow-moving baseline. After subtraction:

$$
\mathbf{x}[t] = \mathbf{y}[t] - \mathbf{d}[t], \quad \mathbf{d}[t] = \alpha\,\mathbf{d}[t-1] + (1-\alpha)\,\mathbf{y}[t]
$$

where $\alpha \in [0.95, 0.99]$ controls the baseline time constant.


In [ ]:
# Core BLE listener — simplified for writeup
import asyncio
from bleak import BleakClient

SENSOR_UUID = "00002a58-0000-1000-8000-00805f9b34fb"

class SensorStream:
    def __init__(self, address: str, alpha: float = 0.97):
        self.address = address
        self.alpha = alpha
        self.queue: asyncio.Queue = asyncio.Queue(maxsize=512)
        self.baseline = None

    def _notification_handler(self, sender, data: bytearray):
        reading = self._parse(data)
        if self.baseline is None:
            self.baseline = reading
        self.baseline = self.alpha * self.baseline + (1 - self.alpha) * reading
        self.queue.put_nowait(reading - self.baseline)  # drift-corrected

    async def stream(self):
        async with BleakClient(self.address) as client:
            await client.start_notify(SENSOR_UUID, self._notification_handler)
            while True:
                yield await self.queue.get()

    def _parse(self, data: bytearray):
        import numpy as np
        return np.frombuffer(data, dtype=np.float32)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from assets.matplotlib_theme import apply_void_theme, vc
apply_void_theme('dark')

# Simulate drift-corrected sensor signal
np.random.seed(42)
t = np.linspace(0, 4, 1000)
drift = 0.3 * np.sin(0.4 * t) + 0.01 * t
signal = np.sin(2 * np.pi * 2 * t) * np.exp(-0.2 * t) + 0.08 * np.random.randn(len(t))
raw = signal + drift

# EMA baseline
alpha = 0.97
baseline = np.zeros_like(raw)
baseline[0] = raw[0]
for i in range(1, len(raw)):
    baseline[i] = alpha * baseline[i-1] + (1 - alpha) * raw[i]
corrected = raw - baseline

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axes[0].plot(t, raw, color=vc.cyan, lw=1.2, label='raw')
axes[0].plot(t, baseline, color=vc.orange, lw=1.2, ls='--', label='baseline (α=0.97)')
axes[0].set_ylabel('$\\hat{Y}$ (normalized)')
axes[0].legend()
axes[0].set_title('Capacitive Sensor — Drift Correction')

axes[1].plot(t, corrected, color=vc.green, lw=1.2)
axes[1].axhline(0, color=vc.muted, lw=0.6, ls=':')
axes[1].set_ylabel('$\\mathbf{x}[t]$ (corrected)')
axes[1].set_xlabel('time (s)')

plt.tight_layout()
plt.show()

## FastAPI Endpoints

```python
from fastapi import FastAPI
from fastapi.responses import StreamingResponse

app = FastAPI()
sensor = SensorStream(address="XX:XX:XX:XX:XX:XX")

@app.get("/stream")
async def stream_readings():
    async def event_generator():
        async for reading in sensor.stream():
            yield f"data: {reading.tolist()}\n\n"
    return StreamingResponse(event_generator(), media_type="text/event-stream")

@app.get("/latest")
async def get_latest():
    return {"reading": sensor.queue.queue[-1].tolist() if sensor.queue.qsize() else None}
```

::: {.callout-note}
SSE (`text/event-stream`) was chosen over WebSockets because downstream consumers
are read-only dashboards. No need for bidirectional comms.
:::
